In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# Define the input and output directories
input_directory = '/home/ahmedmas/Projects/Data_imputation/Processed_Data'
train_output_directory = '/home/ahmedmas/Projects/Data_imputation/model_result/train'
test_output_directory = '/home/ahmedmas/Projects/Data_imputation/model_result/test'

# Create the output directories if they do not exist
os.makedirs(train_output_directory, exist_ok=True)
os.makedirs(test_output_directory, exist_ok=True)

# Function to extract study site from filename
def extract_study_site(filename):
    parts = filename.split('_')
    study_site = parts[1]  # Assuming the study site is the second word in the filename
    return study_site

# Function to calculate missing percentage
def calculate_missing_percentage(df, column):
    total_values = len(df)
    missing_values = df[column].isnull().sum()
    missing_percentage = (missing_values / total_values) * 100
    return missing_percentage, missing_values, total_values

# Function to split the data into training and testing sets
def split_train_test(df, target_column):
    train_df = df[df[target_column].notnull()]
    test_df = df[df[target_column].isnull()]
    return train_df, test_df

# Function to add hour, day, and month columns
def add_time_columns(df):
    df['Hour'] = df['datetime'].dt.hour
    df['Day'] = df['datetime'].dt.day
    df['Month'] = df['datetime'].dt.month
    return df

# Traverse the directory and process each CSV file
model_name = 'KNN'
for filename in os.listdir(input_directory):
    if filename.endswith('.csv'):
        study_site = extract_study_site(filename)
        filepath = os.path.join(input_directory, filename)

        # Load the CSV file into a DataFrame
        df = pd.read_csv(filepath)

        # Convert datetime column to datetime type
        df['datetime'] = pd.to_datetime(df['datetime'])

        # Calculate initial missing percentage of PM2.5
        initial_missing_percentage, initial_missing_count, total_values = calculate_missing_percentage(df, 'PM2.5')

        # Create a column to mark originally missing data
        df['missing_info'] = np.where(df['PM2.5'].isnull(), 'OM', '')

        # Determine the number of additional values to remove to achieve 10% missing
        target_missing_percentage = 10
        target_missing_count = int(target_missing_percentage * total_values / 100)
        additional_missing_count = target_missing_count - initial_missing_count

        # Create a column to store removed values
        df['removed_values'] = np.nan

        if additional_missing_count > 0:
            # Randomly select indices to set as NaN
            non_missing_indices = df[df['PM2.5'].notnull()].index
            if len(non_missing_indices) < additional_missing_count:
                print(f"Not enough non-missing values in {filename} to achieve 10% missing")
                continue
            additional_missing_indices = np.random.choice(non_missing_indices, additional_missing_count, replace=False)
            df.loc[additional_missing_indices, 'removed_PM2.5'] = df.loc[additional_missing_indices, 'PM2.5']
            df.loc[additional_missing_indices, 'PM2.5'] = np.nan

            # Mark the artificially missing data
            df.loc[additional_missing_indices, 'missing_info'] = 'AM'

        # Add columns for hour, day, and month
        df = add_time_columns(df)

        # Split the data into training and testing sets
        train_df, test_df = split_train_test(df, 'PM2.5')

        # Drop the specified columns from training and testing sets
        columns_to_drop = ['datetime', 'missing_info', 'removed_values']
        train_df_dropped = train_df.drop(columns=columns_to_drop)
        test_df_dropped = test_df.drop(columns=columns_to_drop)

        # Combine training and testing sets for KNN imputation
        combined_df = pd.concat([train_df_dropped, test_df_dropped])

        # Use KNN imputer to predict missing PM2.5 values
        imputer = KNNImputer(n_neighbors=5)
        imputed_df = pd.DataFrame(imputer.fit_transform(combined_df), columns=combined_df.columns)

        # Extract the predicted PM2.5 values for the test set
        test_df['PM2.5'] = imputed_df.loc[test_df_dropped.index, 'PM2.5']

        # For training set, include both observed and predicted PM2.5
        train_df['Predicted_PM2.5'] = imputed_df.loc[train_df.index, 'PM2.5']

        # Save the training and test sets to the output directory
        train_output_filepath = os.path.join(train_output_directory, f'train_{model_name}_{filename}')
        test_output_filepath = os.path.join(test_output_directory, f'test_{model_name}_{filename}')
        train_df.to_csv(train_output_filepath, index=False)
        test_df.to_csv(test_output_filepath, index=False)
        print(f"Processed and saved train file: {train_output_filepath}")
        print(f"Processed and saved test file: {test_output_filepath}")

print("Processing complete.")


/home/ahmedmas/.local/lib/python3.6/site-packages/ipykernel_launcher.py:100: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/home/ahmedmas/.local/lib/python3.6/site-packages/ipykernel_launcher.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Processed and saved train file: /home/ahmedmas/Projects/Data_imputation/model_result/train/train_KNN_AQMS_ARMIDALE_2018-12-31_2024-07-16_data.csv
Processed and saved test file: /home/ahmedmas/Projects/Data_imputation/model_result/test/test_KNN_AQMS_ARMIDALE_2018-12-31_2024-07-16_data.csv


/home/ahmedmas/.local/lib/python3.6/site-packages/ipykernel_launcher.py:100: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/home/ahmedmas/.local/lib/python3.6/site-packages/ipykernel_launcher.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Processed and saved train file: /home/ahmedmas/Projects/Data_imputation/model_result/train/train_KNN_AQMS_BATHURST_2018-12-31_2024-07-16_data.csv
Processed and saved test file: /home/ahmedmas/Projects/Data_imputation/model_result/test/test_KNN_AQMS_BATHURST_2018-12-31_2024-07-16_data.csv


/home/ahmedmas/.local/lib/python3.6/site-packages/ipykernel_launcher.py:100: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/home/ahmedmas/.local/lib/python3.6/site-packages/ipykernel_launcher.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


...

Processed and saved train file: /home/ahmedmas/Projects/Data_imputation/model_result/train/train_KNN_AQMS_PARRAMATTA_2018-12-31_2024-07-16_data.csv
Processed and saved test file: /home/ahmedmas/Projects/Data_imputation/model_result/test/test_KNN_AQMS_PARRAMATTA_2018-12-31_2024-07-16_data.csv


/home/ahmedmas/.local/lib/python3.6/site-packages/ipykernel_launcher.py:100: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/home/ahmedmas/.local/lib/python3.6/site-packages/ipykernel_launcher.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Processed and saved train file: /home/ahmedmas/Projects/Data_imputation/model_result/train/train_KNN_AQMS_WAGGA_2018-12-31_2024-07-16_data.csv
Processed and saved test file: /home/ahmedmas/Projects/Data_imputation/model_result/test/test_KNN_AQMS_WAGGA_2018-12-31_2024-07-16_data.csv


/home/ahmedmas/.local/lib/python3.6/site-packages/ipykernel_launcher.py:100: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/home/ahmedmas/.local/lib/python3.6/site-packages/ipykernel_launcher.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Processed and saved train file: /home/ahmedmas/Projects/Data_imputation/model_result/train/train_KNN_AQMS_WOLLONGONG_2018-12-31_2024-07-16_data.csv
Processed and saved test file: /home/ahmedmas/Projects/Data_imputation/model_result/test/test_KNN_AQMS_WOLLONGONG_2018-12-31_2024-07-16_data.csv
Processing complete.
